# Задание 1

#### Функция расчёта размера выборки для A/B-теста

$n > \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2 \cdot V(x)}{MDE^2}$


In [180]:
import math
from scipy import stats

def calc_sample_size(mde, var, alpha, power): # power = 1 - beta
  z_alpha = round(stats.norm.ppf(1 - alpha/2), 2) # лучше без округления
  z_beta = round(stats.norm.ppf(power), 2) # лучше без округления
  n = ((z_alpha + z_beta) ** 2) * 2 * var / (mde ** 2)
  n = math.ceil(n)
  print(f'При дисперсии метрики {var} и необходимости зафиксировать эффект от {mde} (MDE), для достижения мощности 1-β = {power} и уровня значимости α = {alpha} требуется {n} участников в каждой группе A/B‑теста.')

#### Пример работы функции:

In [181]:
mde1 = 0.020
var1= 0.090
alpha1 = 0.05
power1 = 0.80
result1 = calc_sample_size(mde1, var1, alpha1, power1)

При дисперсии метрики 0.09 и необходимости зафиксировать эффект от 0.02 (MDE), для достижения мощности 1-β = 0.8 и уровня значимости α = 0.05 требуется 3528 участников в каждой группе A/B‑теста.


# Задание 2

**Кейс:** онлайн-платформа для обучения

**Проблема:** мало студентов заканчивают первый модуль курсов по мат. статистике

**Гипотеза:** если новым студентам отправлять мотивационное уведомление через 2 дня после старта обвучения на курсе, то доля студентов, завершивших курс, увеличится

**Описание трафика:** каждый день курс по мат. статистике начинают проходить 280 человек

#### Расчёт размера выборки: 1568 в группе

In [182]:
mde2 = 0.05
var2= 0.25
alpha2 = 0.05
beta2 = 0.2
power2 = 1 - beta2
result2 = calc_sample_size(mde2, var2, alpha2, power2)

При дисперсии метрики 0.25 и необходимости зафиксировать эффект от 0.05 (MDE), для достижения мощности 1-β = 0.8 и уровня значимости α = 0.05 требуется 1568 участников в каждой группе A/B‑теста.


#### Расчёт длительности эксперимента:

In [183]:
sample_size = 1568
traffic = 280
additional_days = 2

duration = 2*sample_size / traffic + additional_days
print(f'Длительность (дн.): {math.ceil(duration)}')

Длительность (дн.): 14


Размер выборки для A/B теста составляет 1568 пользователей на группу (всего 3136 пользователей).

При ежедневном притоке 280 новых студентов на курс **необходимое время набора выборки**: 3136 / 280 ≈ **12 дней**.

Метрика требует дополнительного времени для проявления эффекта (уведомление отправляется через 2 дня после старта обучения), поэтому добавляем 2 дня. Итого минимальная длительность эксперимента составляет ≈ 14 дней (округление вверх).

Дополнительно необходимо учитывать недельную «сезонность» поведения пользователей. Продолжительность теста в 14 дней покрывает два полных недельных цикла, что делает результаты устойчивыми к внутри-недельным колебаниям активности.

Но так как основная метрика — это завершение модуля, то нужно время, чтобы студент успел его завершить, обычно на это нужно больше, чем 1 день. Если первый модуль проходится, например, 5–7 дней, то добавлять нужно ещё и это время, а не только момент отправки уведомления.

Следовательно, рекомендуемая длительность A/B теста — **14 дней + среднее время прохождения модуля**.

# Задание 3

#### Функция, которая разделяет пользователей в тестовую или контрольную группы при первом заходе в приложение/систему.

Простая функция, рандомно распределяющая пользователя в группу (без проверки одинаковости размера выборок):

In [184]:
# Вариант 1
import random

assignmnts = {}

def assign_user_to_group(user_id, experiment_id, groups=('A', 'B')):
  key = (user_id, experiment_id)

  if key in assignmnts:
    return assignmnts[key]

  group = random.choice(groups)
  assignmnts[key] = group
  return group

Генерация тестового датасета:

In [185]:
import pandas as pd
import numpy as np

np.random.seed(77)

# создаём список пользователей и экспериментов
users = [f'user_{i}' for i in range(1, 51)]
experiments = ['exp_1', 'exp_2']

n_events = 100

# случайные первые заходы
df_events = pd.DataFrame({
    'user_id': np.random.choice(users, size=n_events),
    'experiment_id': np.random.choice(experiments, size=n_events)
})

df_events.head()

,user_id,experiment_id
0,user_24,exp_1
1,user_32,exp_2
2,user_21,exp_1
3,user_21,exp_1
4,user_44,exp_2


Применение функции:

In [186]:
df_events['group'] = df_events.apply(lambda row: assign_user_to_group(row['user_id'], row['experiment_id']), axis=1)

df_events.head()

,user_id,experiment_id,group
0,user_24,exp_1,A
1,user_32,exp_2,A
2,user_21,exp_1,A
3,user_21,exp_1,A
4,user_44,exp_2,B


Проверка баланса количества пользователей в группах:

In [187]:
assignmnts_df = pd.DataFrame([{'user_id': key[0], 'experiment_id': key[1], 'group': value}
                              for key, value in assignmnts.items()])

assignmnts_df.groupby(['experiment_id', 'group']).size()

experiment_id  group
exp_1          A        19
               B        14
exp_2          A        13
               B        20
dtype: int64

Можно заметить, что сейчас группы не одинаковые по размеру, поэтому я решила усовершенствовать функцию проверкой на одинаковость с помощью ИИ.

Чтобы обеспечить распределение 50/50 с контролем дисбаланса, логика назначения должна учитывать текущее количество пользователей в каждой группе внутри конкретного эксперимента. Если пользователь уже был распределён — возвращаем прежнюю группу. Если новый — отправляем в ту группу, где сейчас меньше наблюдений (или случайно, если равенство).

In [188]:
# Вариант 2
assignments = pd.DataFrame(columns=['user_id', 'experiment_id', 'group'])


def assign_group(user_id, experiment_id):
  global assignments

  # проверяем, есть ли уже назначение
  existing = assignments[(assignments['user_id'] == user_id) & (assignments['experiment_id'] == experiment_id)]
  if not existing.empty:
    return existing.iloc[0]['group']

  # считаем текущий баланс по эксперименту
  exp_data = assignments[assignments['experiment_id'] == experiment_id]
  counts = exp_data['group'].value_counts()

  count_A = counts.get('A', 0)
  count_B = counts.get('B', 0)

  # назначаем в меньшую группу
  if count_A < count_B:
    group = 'A'
  elif count_B < count_A:
    group = 'B'
  else:
    group = np.random.choice(['A', 'B'])

  # сохраняем назначение
  assignments = pd.concat([assignments, pd.DataFrame([{'user_id': user_id, 'experiment_id': experiment_id, 'group': group}])], ignore_index=True)

  return group

Применение и проверка баланса:

In [189]:
df_events['group'] = df_events.apply(lambda row: assign_group(row['user_id'], row['experiment_id']), axis=1)

assignments.groupby(['experiment_id', 'group']).size()

experiment_id  group
exp_1          A        16
               B        17
exp_2          A        16
               B        17
dtype: int64

Сейчас группы получаются примерно одинаковыми по количеству пользователей в них.



# Задание 4: Рефлексия использования ИИ

В первом и втором заданиях я использовала ИИ только для проверки кода, который написала сама, чтобы проанализировать корректность, эффективность и верность решений, потому что хотелось разобраться в деталях, которые потом часто будут переиспользоваться мной в других AB-тестах (+ здания достаточно простые). В третьем задании я попросила ChatGPT переделать мой вариант кода так, чтобы функция распределяла пользователей равномерно между группами A и B, тогда как мой вариант полагался только на одинаковую вероятность выбора одной из двух групп в методе numpy.random.choice(), которая не даёт абсолютного баланса и равных групп.

Результатом я удовлетворена, но всё равно много чего проверяла с помощью документаций библиотек вручную, перечитывала весь код, анализировала, как бы сделала сама, и меняла или не использовала то, что мне казалось избыточным (например, использование библиотеки hashlib, генерацию хешированных значений и определение группы по ним).